# Latent-state HMM analysis — unsupervised neural state modelling

This notebook complements the supervised population-geometry notebook.

## Motivation
The supervised analyses show that behaviour-linked population structure exists.
But the scientific question for this project is stronger:

> Can the neural population itself reveal **latent internal states**, without
> imposing behavioural labels first?

This notebook uses a Gaussian HMM on a PCA-reduced, standardised neural
representation to answer that question.

## Main goals
1. Identify a small number of latent neural states from continuous population activity
2. Quantify how those states align with attack / submission / sniff / phase structure
3. Examine whether Pre → Post remapping is visible as a change in latent-state occupancy
4. Provide a state-sequence view of the session, rather than only behaviour-conditioned decoding

## What was fixed vs. the original notebook
| Issue | Fix |
|---|---|
| External `utils.py` dependency | All helpers inlined — fully standalone |
| Hardcoded data path | `DATA_DIR` variable at top of Section 1 |
| HMM input not standardised | `StandardScaler` applied to PCA scores before HMM |
| Single random initialisation | Multi-start (5 seeds), best log-likelihood selected |
| No convergence check | `monitor_.converged` checked; warning if not converged |
| BIC only, no AIC | Both AIC and BIC proxy computed |
| Hard Viterbi labels only | Soft (posterior) behaviour enrichment added alongside hard |
| PCA fit on 9 mean vectors for overlay | PCA fit on full time-series; centroids projected in |
| No state mean visualisation | Heatmap of emission means in neural (z-score) space |
| No Pre→Post occupancy test | Bootstrap confidence intervals on Pre vs Post difference |
| No baseline-corrected attack PETH | Baseline (pre-onset mean) subtracted in PETH |
| State label order arbitrary | States reordered by overall occupancy for readability |
| N_PCS = 80% variance threshold (67) | N_PCS swept systematically; 20 used as default |
| BIC/AIC range K=3..10 only | Extended to K=3..15; K=9 sensitivity check added |
| State 6 (attack-enriched) not analysed | Deep-dive cell with centroid distances and PETH |

## Important interpretation notes
- This is an **unsupervised** state model — state labels are descriptive, not ground truth.
- A Gaussian HMM is a simple first model, not a full dynamical-systems solution.
- State-number selection should be treated as heuristic / model-based, not definitive biology.
- Temporal autocorrelation in neural data means the effective sample size for
  BIC/AIC is much less than the number of time bins.

In [ ]:
# =============================================================================
# SECTION 0 · Imports
# =============================================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.distance import cdist
from scipy import stats

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    from hmmlearn.hmm import GaussianHMM
    HMM_AVAILABLE = True
except Exception as e:
    HMM_AVAILABLE = False
    HMM_IMPORT_ERROR = repr(e)
    print(f"⚠  hmmlearn not available: {HMM_IMPORT_ERROR}")
    print("   Install with:  pip install hmmlearn")

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"HMM available: {HMM_AVAILABLE}")

## 1. Load dataset

Set `DATA_DIR` below to the folder containing the three `.mat` files.
No external `utils.py` required — fully standalone.

In [ ]:
# =============================================================================
# SECTION 1 · Data loading  (standalone, no utils.py)
# =============================================================================

DATA_DIR = Path(".")   # ← change to your data folder if needed


def _to_str(x):
    if isinstance(x, bytes):
        return x.decode("utf-8")
    if isinstance(x, np.ndarray) and x.size == 1:
        return _to_str(x.item())
    return str(x)


def cell_to_dict(cell_like):
    """Convert MATLAB cell array (N×2: {name, value}) to Python dict."""
    arr = np.asarray(cell_like, dtype=object)
    arr = np.squeeze(arr)
    out = {}
    if arr.ndim == 2 and arr.shape[-1] == 2:
        iterable = arr
    elif arr.ndim == 1 and len(arr) > 0 and isinstance(arr[0], (list, tuple, np.ndarray)):
        iterable = arr
    else:
        raise ValueError(f"Unsupported cell array shape: {arr.shape}")
    for row in iterable:
        name  = _to_str(row[0])
        value = np.asarray(row[1]).squeeze()
        out[name] = (value.astype(float)
                     if np.issubdtype(np.asarray(value).dtype, np.number)
                     else value)
    return out


def smooth_and_zscore_fr(FR, fs=50.0, smooth_width_ms=200.0):
    smoothWidth_bins = round((smooth_width_ms / 1000.0) * fs)  # = 10
    sigma_bins = smoothWidth_bins / 3                           # = 3.0，和MATLAB一致
    FR_smooth = gaussian_filter1d(FR.astype(float), sigma=sigma_bins,
                                  axis=1, mode="nearest")
    mu = FR_smooth.mean(axis=1, keepdims=True)
    sd = FR_smooth.std(axis=1,  keepdims=True)
    sd[sd == 0] = 1.0
    FRz = (FR_smooth - mu) / sd
    FRz[~np.isfinite(FRz)] = 0.0
    return FR_smooth, FRz


def build_phase_masks(n_timebins, fs=50.0,
                      gate_open_time_s=5 * 60 + 45,
                      white_duration_s=5 * 60,
                      dark_duration_s=5 * 60):
    n         = int(n_timebins)
    pre_end   = min(int(round(gate_open_time_s * fs)), n)
    white_end = min(int(round((gate_open_time_s + white_duration_s) * fs)), n)
    dark_end  = min(int(round((gate_open_time_s + white_duration_s + dark_duration_s) * fs)), n)
    pre = np.zeros(n, bool); pre[:pre_end]              = True
    wh  = np.zeros(n, bool); wh[pre_end:white_end]      = True
    dk  = np.zeros(n, bool); dk[white_end:dark_end]     = True
    po  = np.zeros(n, bool); po[dark_end:]              = True
    return {"Pre": pre, "WhiteAgg": wh, "DarkAgg": dk, "Post": po}


# ── Load ─────────────────────────────────────────────────────────────
N = sio.loadmat(DATA_DIR / "neuralData.mat")
P = sio.loadmat(DATA_DIR / "peripheralData.mat")
B = sio.loadmat(DATA_DIR / "behaviorData.mat")

FR     = np.asarray(N["FR"], dtype=float)
periph = cell_to_dict(P["peripheralData"])
behav  = cell_to_dict(B["behaviorData"])

fs = 50.0
dt = 1.0 / fs
n_neurons, n_timebins = FR.shape
t_min = np.arange(n_timebins) / fs / 60.0

print(f"Neural data: {n_neurons} neurons × {n_timebins} bins "
      f"({n_timebins/fs/60:.1f} min at {fs} Hz)")

# ── Signals ───────────────────────────────────────────────────────────
gate_open   = periph["gateOpen"].astype(bool)
white_light = periph["whiteLight"].astype(bool)
sniff_urine = periph["ctBudSniff"].astype(bool)
lick_milk   = periph["lickMilk"].astype(bool)
lick_empty  = periph["lickEmpty"].astype(bool)

attack_left   = behav["attackOfIntLeft"].astype(bool)
attack_right  = behav["attackOfIntRight"].astype(bool)
attack_dark   = behav["attackDark"].astype(bool)
attack        = attack_left | attack_right | attack_dark

submission_left  = behav["submissionSignFromIntLeft"].astype(bool)
submission_right = behav["submissionSignFromIntRight"].astype(bool)
submission       = submission_left | submission_right

chase_left  = behav.get("chaseOfIntLeft",  np.zeros(n_timebins, bool)).astype(bool)
chase_right = behav.get("chaseOfIntRight", np.zeros(n_timebins, bool)).astype(bool)
chase       = chase_left | chase_right

phase_masks = build_phase_masks(n_timebins, fs=fs)

behaviour_masks = {
    "attack":      attack,
    "submission":  submission,
    "chase":       chase,
    "sniff_urine": sniff_urine,
    "lick_milk":   lick_milk,
    "lick_empty":  lick_empty,
    "white_light": white_light,
}

# ── Preprocessing ─────────────────────────────────────────────────────
FR_smooth, FRz = smooth_and_zscore_fr(FR, fs=fs, smooth_width_ms=200)
print("FRz shape:", FRz.shape)
print("Phase bin counts:", {k: int(v.sum()) for k, v in phase_masks.items()})

## 2. Helper functions — geometry, HMM utilities, enrichment

In [ ]:
# =============================================================================
# SECTION 2 · Helper functions
# =============================================================================

# ─── geometry ────────────────────────────────────────────────────────────────

def condition_pattern(FRz, mask):
    mask = np.asarray(mask).astype(bool)
    if mask.sum() == 0:
        return np.zeros(FRz.shape[0])
    return FRz[:, mask].mean(axis=1)


def pairwise_distance_tables(centroid_dict):
    keys = list(centroid_dict.keys())
    X    = np.stack([centroid_dict[k] for k in keys], axis=0)
    eu   = pd.DataFrame(cdist(X, X, metric="euclidean"),   index=keys, columns=keys)
    coss = pd.DataFrame(1 - cdist(X, X, metric="cosine"), index=keys, columns=keys)
    return eu, coss


# ─── autocorrelation time (Sokal windowed estimator) ─────────────────────────

def _acf_fft(x, max_lag):
    """FFT-based normalised autocorrelation up to max_lag."""
    x = x - x.mean()
    n = len(x)
    xp  = np.concatenate([x, np.zeros(n)])
    f   = np.fft.rfft(xp)
    acf = np.fft.irfft(f * np.conj(f))[:max_lag + 1]
    if acf[0] == 0:
        return np.zeros(max_lag + 1)
    return acf / acf[0]


def compute_autocorr_time(x, max_lag=2000):
    """
    Integrated autocorrelation time τ_int = 1 + 2·Σ_{k=1}^{M} ρ(k)
    where M is Sokal's automatic window (first lag at which ρ(k) ≤ 0).
    Uses FFT for speed.  Returns τ in bins.

    NOTE: Do NOT call this on raw PC scores if the session contains slow
    experimental-phase transitions — those inflate τ enormously (≫ dwell
    times) and make n_eff useless.  Use compute_phase_detrended_tau()
    instead.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    if np.var(x) == 0:
        return 1.0
    max_lag = min(max_lag, n - 1)
    acf = _acf_fft(x, max_lag)
    zero_cross = np.where(acf[1:] <= 0)[0]
    cutoff = (zero_cross[0] + 1) if len(zero_cross) > 0 else max_lag
    return max(1.0 + 2.0 * float(acf[1:cutoff].sum()), 1.0)


def compute_phase_detrended_tau(X_pca, phase_masks, n_pcs=5, max_lag=2000):
    """
    Estimate τ_int from *phase-detrended* PC scores.

    Raw PC1 is dominated by the slow Pre→WhiteAgg→DarkAgg→Post trend,
    giving τ ≈ 60-70 s and n_eff ≈ 18 — useless for BIC.  The relevant
    timescale for HMM state-switching is the within-phase neural dynamics
    (characteristic dwell time ~2–4 s).

    Detrending: subtract each phase's mean from its time bins, leaving
    only within-phase fluctuations.  Then apply the Sokal estimator.

    Returns
    -------
    tau_by_pc : dict  {pc_index: τ (bins)}
    X_detrended : ndarray  (n_timebins × n_pcs)  detrended scores
    acf_pc0     : ndarray  normalised ACF of detrended PC1 (for plotting)
    """
    X_dt = X_pca[:, :n_pcs].copy()
    for pm in phase_masks.values():
        pm = np.asarray(pm).astype(bool)
        if pm.sum() > 0:
            X_dt[pm] -= X_dt[pm].mean(axis=0)

    tau_by_pc = {}
    for pc_i in range(n_pcs):
        tau_by_pc[pc_i] = compute_autocorr_time(X_dt[:, pc_i], max_lag=max_lag)

    acf_pc0 = _acf_fft(X_dt[:, 0], min(max_lag, len(X_dt) - 1))
    return tau_by_pc, X_dt, acf_pc0


# ─── HMM model-selection metrics (raw) ───────────────────────────────────────

def hmm_bic_aic(logL, n_states, n_features, n_samples):
    """
    BIC and AIC for a Gaussian HMM with diagonal covariance.
    Parameter count: (K-1) + K*(K-1) + 2*K*d   (K states, d features).
    NOTE: uses n_samples = n_timebins — gives MONOTONE BIC for
    autocorrelated neural data.  Use hmm_bic_aic_corrected() instead.
    """
    K, d = n_states, n_features
    n_params = (K - 1) + K * (K - 1) + 2 * K * d
    bic = -2 * logL + n_params * np.log(max(n_samples, 2))
    aic = -2 * logL + 2 * n_params
    return bic, aic, n_params


# ─── HMM model-selection metrics (autocorrelation-corrected) ─────────────────

def hmm_bic_aic_corrected(logL, n_states, n_features, n_samples, tau):
    """
    Autocorrelation-corrected BIC for a Gaussian HMM with diagonal covariance.

    Temporal autocorrelation in neural data means the effective sample size
    n_eff = n_timebins / τ, where τ is the integrated autocorrelation time
    (bins, estimated by compute_autocorr_time).  Substituting n_eff in the
    BIC log-penalty produces a finite elbow even when the raw BIC is monotone.

    The AIC has no log-n term so it is unchanged by this correction.

    Reference: Thiébaux & Zwiers (1984, Mon. Weather Rev.);
               Sokal (1989) for τ_int estimation.
    """
    K, d  = n_states, n_features
    n_eff = max(n_samples / tau, 2.0)
    n_params  = (K - 1) + K * (K - 1) + 2 * K * d
    bic_corr  = -2 * logL + n_params * np.log(n_eff)
    aic_corr  = -2 * logL + 2 * n_params   # AIC unchanged
    return bic_corr, aic_corr, n_params

# ─── HMM fitting with multi-start ────────────────────────────────────────────

def fit_hmm_multistart(X, n_states, n_starts=5, n_iter=300,
                       covariance_type="diag"):
    """
    Fit a Gaussian HMM using multiple random initialisations; return the best
    (highest log-likelihood) model.  Convergence is checked and reported.
    """
    best_model = None
    best_logL  = -np.inf
    for seed in range(n_starts):
        hmm = GaussianHMM(
            n_components=n_states,
            covariance_type=covariance_type,
            n_iter=n_iter,
            random_state=seed,
        )
        try:
            hmm.fit(X)
            logL = hmm.score(X)
            if logL > best_logL:
                best_logL  = logL
                best_model = hmm
        except Exception:
            continue
    if best_model is None:
        raise RuntimeError(f"All {n_starts} HMM fits failed for K={n_states}.")
    if not best_model.monitor_.converged:
        print(f"  ⚠  K={n_states}: best model did NOT converge in {n_iter} iterations.")
    return best_model, best_logL


# ─── state ordering ──────────────────────────────────────────────────────────

def reorder_states_by_occupancy(state_seq, K):
    """Relabel so state 0 is the most occupied, state K-1 the least."""
    counts  = np.bincount(state_seq, minlength=K)
    order   = np.argsort(-counts)
    mapping = {old: new for new, old in enumerate(order)}
    new_seq = np.array([mapping[s] for s in state_seq], dtype=int)
    return new_seq, mapping


# ─── behaviour enrichment ────────────────────────────────────────────────────

def behaviour_enrichment_hard(state_seq, masks_dict):
    """Hard (Viterbi) behaviour enrichment."""
    rows = []
    for s in np.unique(state_seq):
        st = state_seq == s
        for nm, mask in masks_dict.items():
            rows.append({
                "state":             int(s),
                "variable":          nm,
                "fraction_in_state": float(mask[st].mean()) if st.sum() > 0 else np.nan,
                "fraction_of_var":   float((mask & st).sum() / max(mask.sum(), 1)),
            })
    return pd.DataFrame(rows)


def behaviour_enrichment_soft(state_post, masks_dict):
    """Soft (posterior) behaviour enrichment."""
    rows = []
    K = state_post.shape[1]
    for nm, mask in masks_dict.items():
        mask = np.asarray(mask).astype(bool)
        if mask.sum() == 0:
            continue
        mean_post = state_post[mask].mean(axis=0)
        for s in range(K):
            rows.append({
                "state":    s,
                "variable": nm,
                "mean_posterior_given_behaviour": float(mean_post[s]),
            })
    return pd.DataFrame(rows)


# ─── dwell times ─────────────────────────────────────────────────────────────

def contiguous_runs(x):
    x = np.asarray(x)
    if len(x) == 0:
        return []
    runs, start = [], 0
    for i in range(1, len(x)):
        if x[i] != x[i - 1]:
            runs.append((x[start], start, i - 1)); start = i
    runs.append((x[start], start, len(x) - 1))
    return runs


def state_dwell_summary(state_seq, fs):
    rows = []
    for s, a, b in contiguous_runs(state_seq):
        rows.append({"state": int(s), "start_bin": int(a),
                     "end_bin": int(b), "duration_s": (b - a + 1) / fs})
    if not rows:
        return pd.DataFrame(), pd.DataFrame()
    dwell_df = pd.DataFrame(rows)
    summary  = (dwell_df.groupby("state")["duration_s"]
                .agg(["count", "mean", "median", "std", "sum"])
                .reset_index())
    return dwell_df, summary


# ─── phase occupancy + bootstrap Pre→Post test ───────────────────────────────

def state_occupancy_by_phase(state_seq, phase_masks):
    rows = []
    for phase_name, pm in phase_masks.items():
        seq_phase = state_seq[pm]
        n = len(seq_phase)
        if n == 0: continue
        for s in np.unique(state_seq):
            rows.append({
                "phase":              phase_name,
                "state":              int(s),
                "occupancy_fraction": float((seq_phase == s).mean()),
                "n_bins":             int((seq_phase == s).sum()),
            })
    return pd.DataFrame(rows)


def bootstrap_prepost_diff(state_seq, phase_masks, K, n_boot=1000, seed=0):
    """Bootstrap 95% CIs for (Post − Pre) occupancy fraction per state."""
    rng       = np.random.default_rng(seed)
    pre_bins  = np.flatnonzero(phase_masks["Pre"])
    post_bins = np.flatnonzero(phase_masks["Post"])
    diffs_boot = np.zeros((n_boot, K))
    for b in range(n_boot):
        pre_s  = rng.choice(pre_bins,  size=len(pre_bins),  replace=True)
        post_s = rng.choice(post_bins, size=len(post_bins), replace=True)
        for s in range(K):
            diffs_boot[b, s] = (
                (state_seq[post_s] == s).mean() -
                (state_seq[pre_s]  == s).mean()
            )
    obs_diff = np.array([
        (state_seq[post_bins] == s).mean() - (state_seq[pre_bins] == s).mean()
        for s in range(K)
    ])
    ci_lo = np.percentile(diffs_boot, 2.5,  axis=0)
    ci_hi = np.percentile(diffs_boot, 97.5, axis=0)
    return pd.DataFrame({
        "state":       np.arange(K),
        "obs_diff":    obs_diff,
        "ci_lo":       ci_lo,
        "ci_hi":       ci_hi,
        "significant": (ci_lo > 0) | (ci_hi < 0),
    })


# ─── attack-aligned state PETH ───────────────────────────────────────────────

def attack_state_peth(state_seq, state_post, K, onsets,
                      pre_bins, post_bins, baseline_bins=None):
    """
    Peri-event histogram of latent state occupancy around attack onsets.
    Returns baseline-corrected hard (Viterbi) and soft (posterior) PETHs.
    """
    n_time = pre_bins + post_bins + 1
    if len(onsets) == 0:
        return np.zeros((K, n_time)), np.zeros((K, n_time))
    hard_wins, soft_wins = [], []
    for o in onsets.astype(int):
        hard_wins.append(
            (state_seq[o - pre_bins: o + post_bins + 1][:, None]
             == np.arange(K)).astype(float))
        soft_wins.append(state_post[o - pre_bins: o + post_bins + 1])
    hard_peth = np.stack(hard_wins, axis=0).mean(axis=0).T
    soft_peth = np.stack(soft_wins, axis=0).mean(axis=0).T
    if baseline_bins is None:
        baseline_bins = pre_bins
    hard_bl = hard_peth[:, :baseline_bins].mean(axis=1, keepdims=True)
    soft_bl = soft_peth[:, :baseline_bins].mean(axis=1, keepdims=True)
    return hard_peth - hard_bl, soft_peth - soft_bl


def raw_peth_single_state(state_seq, state_post, target_state, onsets,
                           pre_bins, post_bins):
    """
    Raw (non-baseline-corrected) PETH for one specific state.
    Returns mean hard occupancy and mean soft posterior.
    """
    if len(onsets) == 0:
        n = pre_bins + post_bins + 1
        return np.zeros(n), np.zeros(n)
    hard_wins = [
        (state_seq[o - pre_bins: o + post_bins + 1] == target_state).astype(float)
        for o in onsets.astype(int)
    ]
    soft_wins = [
        state_post[o - pre_bins: o + post_bins + 1, target_state]
        for o in onsets.astype(int)
    ]
    return np.stack(hard_wins).mean(0), np.stack(soft_wins).mean(0)




# ─── K-selection helpers: cross-validation & state stability ──────────────────

def hmm_cv_logL(X, n_states, n_folds=5, n_starts=5, n_iter=300,
                covariance_type="diag", random_state=0):
    """
    Temporal k-fold cross-validation for a Gaussian HMM.

    The session is split into n_folds contiguous chunks.  For each fold,
    the model is trained on the remaining chunks and evaluated on the held-
    out chunk.  Returns the mean held-out log-likelihood per time-bin
    (so values are comparable across different K).

    Why temporal folds?  Shuffling breaks the Markov structure.  Contiguous
    folds preserve it, at the cost of some train/test distribution shift at
    the fold boundaries — acceptable for a single-session heuristic.
    """
    n = len(X)
    fold_size = n // n_folds
    held_out_ll = []
    for fold in range(n_folds):
        lo, hi = fold * fold_size, (fold + 1) * fold_size
        # train mask: everything outside [lo, hi)
        idx_train = np.concatenate([np.arange(0, lo), np.arange(hi, n)])
        X_train = X[idx_train]
        X_test  = X[lo:hi]
        if len(X_train) < n_states * 2 or len(X_test) == 0:
            continue
        try:
            m, _ = fit_hmm_multistart(X_train, n_states,
                                      n_starts=n_starts, n_iter=n_iter,
                                      covariance_type=covariance_type)
            held_out_ll.append(m.score(X_test) / len(X_test))
        except Exception:
            continue
    return float(np.mean(held_out_ll)) if held_out_ll else -np.inf


def hmm_state_stability(X, n_states, n_runs=10, n_iter=300,
                        covariance_type="diag"):
    """
    State-sequence stability across random initialisations.

    Fits the same K-state HMM n_runs times with different random seeds.
    Computes the mean pairwise Adjusted Rand Index (ARI) between all pairs
    of decoded Viterbi sequences.

    ARI = 1.0  → all runs produce identical state assignments (very stable)
    ARI ≈ 0.0  → random-level agreement (K is too large / ill-posed)

    The ARI is alignment-free: it compares clusterings as partitions, so
    arbitrary label permutations do not affect the score.
    """
    from sklearn.metrics import adjusted_rand_score
    seqs = []
    for seed in range(n_runs):
        hmm = GaussianHMM(n_components=n_states,
                          covariance_type=covariance_type,
                          n_iter=n_iter, random_state=seed)
        try:
            hmm.fit(X)
            seqs.append(hmm.predict(X))
        except Exception:
            continue
    if len(seqs) < 2:
        return np.nan
    aris = []
    for i in range(len(seqs)):
        for j in range(i + 1, len(seqs)):
            aris.append(adjusted_rand_score(seqs[i], seqs[j]))
    return float(np.mean(aris))

print("All helpers loaded.")

## 3. PCA dimensionality reduction for latent modelling

**Why PCA first?** Fitting a Gaussian HMM directly in ~200-dimensional neural
space is computationally prohibitive and statistically ill-posed.

**Why standardise after PCA?** PCA components have *decreasing* variance by
construction. Without `StandardScaler`, PC1 dominates the Gaussian emissions and
the HMM effectively fits only to PC1.

**N_PCS choice**: Using the 80% variance threshold gives N_PCS=67, which yields
1439 parameters at K=10. With that many parameters BIC/AIC never flatten —
Cell A below sweeps N_PCS systematically to find a value where an elbow exists.
We start with N_PCS=20 as a reasonable default.

In [ ]:
# =============================================================================
# SECTION 3 · PCA + standardisation
# =============================================================================

X_full = FRz.T   # (n_timebins × n_neurons)

pca_full = PCA(random_state=0)
pca_full.fit(X_full)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

n_80 = int(np.searchsorted(cumvar, 0.80)) + 1
n_90 = int(np.searchsorted(cumvar, 0.90)) + 1
print(f"PCs for 80% variance: {n_80}")
print(f"PCs for 90% variance: {n_90}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(np.arange(1, min(len(cumvar) + 1, 101)), cumvar[:100], marker="o", ms=3)
axes[0].axhline(0.80, color="tab:orange", ls="--", label=f"80% → {n_80} PCs")
axes[0].axhline(0.90, color="tab:red",    ls=":",  label=f"90% → {n_90} PCs")
axes[0].axvline(n_80, color="tab:orange", ls="--", alpha=0.5)
axes[0].axvline(n_90, color="tab:red",    ls=":",  alpha=0.5)
axes[0].set_xlabel("Number of PCs"); axes[0].set_ylabel("Cumulative explained variance")
axes[0].set_title("PCA dimensionality curve"); axes[0].legend()

axes[1].bar(np.arange(1, 21),
            pca_full.explained_variance_ratio_[:20] * 100, color="steelblue", alpha=0.8)
axes[1].set_xlabel("PC"); axes[1].set_ylabel("% Variance explained")
axes[1].set_title("Variance per PC (first 20)")
plt.tight_layout(); plt.show()

In [ ]:
# MODIFIED: N_PCS = 20 instead of n_80 (= 67).
# n_80=67 → 1439 params at K=10; BIC/AIC never flatten within K=3..10.
# N_PCS=20 gives ~200 params at K=10, where an elbow typically emerges.
# Cell A below sweeps [10, 15, 20, 30] to confirm the right choice.
N_PCS = 20

pca_hmm   = PCA(n_components=N_PCS, random_state=0)
X_pca_raw = pca_hmm.fit_transform(X_full)

scaler  = StandardScaler()
X_pca   = scaler.fit_transform(X_pca_raw)

print(f"Using N_PCS = {N_PCS}  (covers "
      f"{cumvar[N_PCS-1]*100:.1f}% of variance)")
print(f"X_pca shape: {X_pca.shape}")
print(f"PC score std before scaling: {X_pca_raw.std(axis=0)[:5].round(3)}")
print(f"PC score std after  scaling: {X_pca.std(axis=0)[:5].round(3)}")

## 3B. Autocorrelation time τ — phase-detrended estimation

**The raw-PC1 problem.**  PC1 encodes the slow experimental phase ramp
(Pre → WhiteAgg → DarkAgg → Post).  Its ACF does not decay within the
session duration, giving τ_raw ≈ 3 000 bins (65 s) and n_eff ≈ 18.
With only 18 effective samples the BIC penalty collapses to
`n_params × log(18) ≈ 2.9 · n_params`, far too weak to penalise
complexity — BIC still monotonically prefers the largest K tested.

**The fix: phase-detrended τ.**  We subtract each phase's time-mean from
its PC scores before computing the Sokal integrated autocorrelation time.
This removes inter-phase structure and leaves only within-phase neural
dynamics, whose characteristic timescale matches state dwell times
(~1–7 s from Section 9).  The resulting n_eff is used in the corrected BIC
in Section 4.


In [ ]:
# =============================================================================
# SECTION 3B · Estimate autocorrelation time τ — phase-detrended approach
# =============================================================================
#
# PROBLEM with raw PC scores
# ──────────────────────────
# PC1 encodes the slow experimental-phase ramp (Pre → WhiteAgg → DarkAgg →
# Post).  Its ACF barely decays within the session, giving
#   τ_raw(PC1) ≈ 3 000–3 500 bins (60–70 s)   →   n_eff ≈ 18
# n_eff = 18 is too small to penalise complexity and makes BIC trivially
# prefer the largest K tested.
#
# SOLUTION: phase-detrended τ
# ────────────────────────────
# Subtract each phase's mean from its time bins.  This removes inter-phase
# trends and leaves only within-phase neural fluctuations — the timescale
# relevant to HMM state switching (cf. mean dwell times ≈ 1–7 s).
#
# References:
#   Sokal (1989) for τ_int; Thiébaux & Zwiers (1984, Mon. Weather Rev.) for
#   effective sample size in autocorrelated series.

tau_raw_by_pc = {}
print("Raw τ (from uncorrected PC scores):")
for pc_i in range(5):
    t = compute_autocorr_time(X_pca[:, pc_i], max_lag=2000)
    tau_raw_by_pc[pc_i] = t
    print(f"  PC{pc_i+1}  τ_raw  = {t:7.1f} bins = {t/fs:.2f} s")

print()
tau_by_pc, X_pca_dt, acf_dt_pc0 = compute_phase_detrended_tau(
    X_pca, phase_masks, n_pcs=5, max_lag=2000)

print("Phase-detrended τ:")
for pc_i, t in tau_by_pc.items():
    print(f"  PC{pc_i+1}  τ_detrend = {t:7.1f} bins = {t/fs:.2f} s")

# Conservative choice: maximum τ across PCs 1-5
TAU_EFF = float(max(tau_by_pc.values()))
N_EFF   = len(X_pca) / TAU_EFF

print(f"\nUsing TAU_EFF = {TAU_EFF:.1f} bins  ({TAU_EFF/fs:.2f} s)")
print(f"→  n_eff = {N_EFF:.0f}  (compression factor = {len(X_pca)/N_EFF:.0f}×)")
print(f"   log(n_eff) = {np.log(N_EFF):.3f}  "
      f"[vs raw log(n) = {np.log(len(X_pca)):.3f}]")
print("(To override: set TAU_EFF = <value> and re-run.)")
# TAU_EFF = 200.0   # ← uncomment to override

# ── Diagnostic plot: raw ACF vs detrended ACF for PC1 ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

# Left: raw PC1 ACF (first 70 s)
n_show = min(int(70 * fs) + 1, len(X_pca) - 1)
acf_raw = _acf_fft(X_pca[:, 0], n_show)
lag_s   = np.arange(len(acf_raw)) / fs
axes[0].plot(lag_s, acf_raw, color="steelblue", lw=1.2)
axes[0].axhline(0,   color="k",         lw=0.6, ls="--")
axes[0].axhline(1/np.e, color="tab:orange", ls=":", label="1/e")
axes[0].axvline(tau_raw_by_pc[0]/fs, color="tab:red", ls="--",
                label=f"τ_raw = {tau_raw_by_pc[0]/fs:.1f} s")
axes[0].set_xlabel("Lag (s)"); axes[0].set_ylabel("Autocorrelation")
axes[0].set_title("PC1 — raw ACF\n(dominated by slow phase trend)")
axes[0].legend(fontsize=9)

# Right: detrended PC1 ACF (first 20 s)
n_show2  = min(int(20 * fs) + 1, len(X_pca) - 1)
acf_dt   = _acf_fft(X_pca_dt[:, 0], n_show2)
lag_s2   = np.arange(len(acf_dt)) / fs
axes[1].plot(lag_s2, acf_dt, color="darkorange", lw=1.2)
axes[1].axhline(0, color="k", lw=0.6, ls="--")
axes[1].axhline(1/np.e, color="tab:orange", ls=":", label="1/e")
axes[1].axvline(tau_by_pc[0]/fs, color="tab:red", ls="--",
                label=f"τ_detrend = {tau_by_pc[0]/fs:.1f} s")
axes[1].set_xlabel("Lag (s)"); axes[1].set_ylabel("Autocorrelation")
axes[1].set_title("PC1 — phase-detrended ACF\n"
                  "(reveals within-phase neural dynamics)")
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()


## 3A. N_PCS sensitivity sweep

Before committing to a K, we need to know whether our choice of N_PCS allows
BIC/AIC to show an elbow. With N_PCS=67 both criteria decrease monotonically
through K=10, so we have no idea where the optimal K is.

This cell sweeps N_PCS ∈ {10, 15, 20, 30} over K=3..8 (quick) and plots the
BIC curves. **Pick the smallest N_PCS that produces a visible elbow**, then
confirm that value is set above (N_PCS=20 in Section 3).

In [ ]:
# =============================================================================
# CELL A · N_PCS sensitivity sweep
# =============================================================================

if not HMM_AVAILABLE:
    print("hmmlearn not available — skipping Cell A.")
else:
    pcs_to_test  = [10, 15, 20, 30]
    k_grid_quick = list(range(3, 9))

    npcs_sweep_rows = []
    print("Sweeping N_PCS × K (3 starts, 200 iter each)…")

    for n_pcs_try in pcs_to_test:
        pca_try   = PCA(n_components=n_pcs_try, random_state=0)
        X_raw_try = pca_try.fit_transform(X_full)
        X_try     = StandardScaler().fit_transform(X_raw_try)
        for k_try in k_grid_quick:
            m, ll = fit_hmm_multistart(X_try, n_states=k_try,
                                       n_starts=3, n_iter=200)
            bic, aic, n_par = hmm_bic_aic(ll, k_try, n_pcs_try, len(X_try))
            npcs_sweep_rows.append({
                "N_PCS": n_pcs_try, "K": k_try, "n_params": n_par,
                "logL": ll, "BIC": bic, "AIC": aic,
                "converged": m.monitor_.converged,
            })
        print(f"  N_PCS={n_pcs_try} done.")

    npcs_sweep_df = pd.DataFrame(npcs_sweep_rows)
    display(npcs_sweep_df.round(1))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for n_pcs_try in pcs_to_test:
        sub = npcs_sweep_df[npcs_sweep_df["N_PCS"] == n_pcs_try]
        axes[0].plot(sub["K"], sub["BIC"],  marker="o", label=f"N_PCS={n_pcs_try}")
        axes[1].plot(sub["K"], sub["logL"], marker="o", label=f"N_PCS={n_pcs_try}")

    axes[0].set_xlabel("K"); axes[0].set_ylabel("BIC (lower = better)")
    axes[0].set_title("BIC vs K for each N_PCS\n← look for elbow →")
    axes[0].legend(fontsize=9)
    axes[1].set_xlabel("K"); axes[1].set_ylabel("Log-likelihood")
    axes[1].set_title("Log-likelihood vs K")
    axes[1].legend(fontsize=9)
    plt.tight_layout(); plt.show()

    print("\n→ If BIC still falls monotonically for all N_PCS values,")
    print("  extend k_grid_quick to range(3, 13) and re-run.")
    print("  If an elbow appears, update N_PCS in Section 3 to that value.")

## 4. HMM model selection — why BIC fails and what to use instead

### Why BIC cannot select K here

For a Gaussian HMM on neural population data, every extra state captures
real variance — the per-state log-likelihood gain (~30 000–50 000 per
additional K) overwhelms any plausible parameter penalty.  Even after
replacing n_timebins with n_eff = n / τ_detrend, the corrected BIC still
decreases monotonically through K = 15.  BIC was designed for i.i.d.
likelihood models; first-order Markov models on high-dimensional,
autocorrelated data violate its assumptions too severely for it to be useful
as a stopping criterion here.

### Two principled alternatives

| Method | What it measures | Key assumption |
|--------|-----------------|----------------|
| **Temporal CV** | Held-out predictive likelihood per bin | A good model generalises to unseen time-windows |
| **State stability (ARI)** | Reproducibility of Viterbi sequences across random seeds | A meaningful K should give consistent state assignments |

Both are computed below over K = 3 … 12.  The **CV elbow** identifies the K
beyond which held-out likelihood stops improving meaningfully.  The **ARI
drop** marks where the solution becomes irreproducible.  The recommended K
is the largest value that still has high ARI *and* lies near the CV elbow.


In [ ]:
# =============================================================================
# SECTION 4 · HMM model selection — CV + ARI (BIC shown for reference only)
# =============================================================================

if not HMM_AVAILABLE:
    raise ImportError("hmmlearn is not installed. Run: pip install hmmlearn")

state_grid = list(range(3, 13))   # K = 3 … 12  (extend if CV/ARI suggest it)

# ── 4a. Fit models + compute BIC (reference only) ────────────────────────────
sel_rows   = []
sel_models = {}

print("Step 1 / 3 — fitting HMMs (5 seeds, 300 iter) …")
for K_sel in state_grid:
    model, logL = fit_hmm_multistart(X_pca, n_states=K_sel,
                                     n_starts=5, n_iter=300)
    bic, aic, n_params = hmm_bic_aic(logL, K_sel, N_PCS, len(X_pca))
    bic_corr, _, _     = hmm_bic_aic_corrected(
                             logL, K_sel, N_PCS, len(X_pca), TAU_EFF)
    conv = model.monitor_.converged
    sel_rows.append({
        "n_states":       K_sel,
        "log_likelihood": logL,
        "n_params":       n_params,
        "BIC":            bic,
        "BIC_corr":       bic_corr,
        "AIC":            aic,
        "converged":      conv,
    })
    sel_models[K_sel] = model
    print(f"  K={K_sel:2d}  logL={logL:.1f}  BIC_corr={bic_corr:.0f}  converged={conv}")

model_sel_df = pd.DataFrame(sel_rows)

# ── 4b. Temporal 5-fold cross-validation ─────────────────────────────────────
print("\nStep 2 / 3 — temporal 5-fold CV (3 starts per fold) …")
print("  (this fits 5 × n_K models — ~5 min on a laptop)")
cv_rows = []
for K_sel in state_grid:
    cv_ll = hmm_cv_logL(X_pca, n_states=K_sel,
                        n_folds=5, n_starts=3, n_iter=200)
    cv_rows.append({"n_states": K_sel, "cv_logL_per_bin": cv_ll})
    print(f"  K={K_sel:2d}  held-out logL/bin = {cv_ll:.4f}")

cv_df = pd.DataFrame(cv_rows)

# ── 4c. State stability (ARI across 10 random seeds) ─────────────────────────
print("\nStep 3 / 3 — state stability (ARI, 10 seeds each) …")
ari_rows = []
for K_sel in state_grid:
    ari = hmm_state_stability(X_pca, n_states=K_sel,
                               n_runs=10, n_iter=200)
    ari_rows.append({"n_states": K_sel, "mean_ARI": ari})
    print(f"  K={K_sel:2d}  mean pairwise ARI = {ari:.3f}")

ari_df = pd.DataFrame(ari_rows)

# Merge all metrics
ksel_df = (model_sel_df[["n_states","log_likelihood","n_params",
                          "BIC","BIC_corr","converged"]]
           .merge(cv_df,  on="n_states")
           .merge(ari_df, on="n_states"))

print("\nFull model-selection table:")
display(ksel_df.round(4))


In [ ]:
# ── 4d. Four-panel model-selection plot ──────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))

Ks = ksel_df["n_states"].values

# Panel 1: log-likelihood
axes[0].plot(Ks, ksel_df["log_likelihood"], marker="o", color="steelblue")
axes[0].set_xlabel("K"); axes[0].set_ylabel("Train log-likelihood")
axes[0].set_title("Train log-likelihood vs K\n(always increases — reference only)")

# Panel 2: BIC (raw + corrected) — for reference
axes[1].plot(Ks, ksel_df["BIC"],      marker="o", color="tab:red",
             label="Raw BIC",  alpha=0.5, ls="--")
axes[1].plot(Ks, ksel_df["BIC_corr"], marker="s", color="tab:purple",
             label=f"Corr BIC (τ={TAU_EFF:.0f})")
axes[1].set_xlabel("K"); axes[1].set_ylabel("BIC (lower = better)")
axes[1].set_title("BIC vs K\n(both monotone — shown for reference only)")
axes[1].legend(fontsize=9)

# Panel 3: CV held-out logL per bin — PRIMARY criterion
axes[2].plot(Ks, ksel_df["cv_logL_per_bin"],
             marker="o", color="tab:green", lw=2)
# Mark elbow: largest K where improvement > 10% of total range
cv_vals = ksel_df["cv_logL_per_bin"].values
cv_diffs = np.diff(cv_vals)
total_range = cv_vals[-1] - cv_vals[0]
threshold = 0.05 * abs(total_range)
meaningful = np.where(cv_diffs > threshold)[0]
K_cv = int(Ks[meaningful[-1] + 1]) if len(meaningful) > 0 else int(Ks[-1])
axes[2].axvline(K_cv, color="black", ls=":", lw=2,
                label=f"CV elbow  K={K_cv}")
axes[2].set_xlabel("K"); axes[2].set_ylabel("Held-out logL / bin")
axes[2].set_title("Temporal CV log-likelihood\n← elbow = diminishing returns")
axes[2].legend(fontsize=10)

# Panel 4: ARI stability — SECONDARY criterion
axes[3].bar(Ks, ksel_df["mean_ARI"],
            color=[("tab:green" if a > 0.6 else
                    "tab:orange" if a > 0.3 else "tab:red")
                   for a in ksel_df["mean_ARI"]],
            alpha=0.85)
axes[3].axhline(0.6, color="k", ls="--", lw=1.2, label="ARI = 0.6 (stable)")
axes[3].axhline(0.3, color="gray", ls=":", lw=1,  label="ARI = 0.3 (marginal)")
axes[3].axvline(K_cv, color="black", ls=":", lw=1.5,
                label=f"CV elbow K={K_cv}")
axes[3].set_xlabel("K"); axes[3].set_ylabel("Mean pairwise ARI")
axes[3].set_title("State stability across seeds\n(ARI < 0.3 = solution irreproducible)")
axes[3].set_ylim(0, 1.05)
axes[3].legend(fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# ── 4e. Final K decision ──────────────────────────────────────────────────────
#
# Selection rule:
#   1. Start from the CV elbow (K_cv): the largest K with meaningful
#      held-out likelihood improvement.
#   2. Cross-check ARI: if ARI at K_cv < 0.3, step down to the largest
#      K where ARI ≥ 0.3.
#   3. The final K is the intersection of these two constraints.

ari_vals = ksel_df.set_index("n_states")["mean_ARI"]

# ARI-stable ceiling: largest K with ARI ≥ 0.3
ari_ok = ksel_df[ksel_df["mean_ARI"] >= 0.3]["n_states"]
K_ari_max = int(ari_ok.max()) if len(ari_ok) > 0 else int(Ks[0])

K_elbow = min(K_cv, K_ari_max)   # ← override here if desired

print("=" * 55)
print(f"CV elbow              K_cv     = {K_cv}")
print(f"ARI-stable ceiling    K_ari    = {K_ari_max}  "
      f"(last K with ARI ≥ 0.3)")
print(f"Selected K            K_elbow  = {K_elbow}")
print(f"  ARI at K_elbow               = "
      f"{float(ari_vals.get(K_elbow, np.nan)):.3f}")
print(f"  CV logL/bin at K_elbow       = "
      f"{float(ksel_df.set_index('n_states')['cv_logL_per_bin'].get(K_elbow, np.nan)):.4f}")
print("=" * 55)

# K_elbow = 8   # ← uncomment to force a specific K

K         = K_elbow
hmm_final = sel_models[K]
print(f"\nFinal model — K={K},  converged={hmm_final.monitor_.converged},  "
      f"train logL={hmm_final.score(X_pca):.1f}")


## 4B. K−1 sensitivity check

Once K is chosen from CV + ARI, this cell fits K−1 to verify that the
least-occupied state is not a split artefact.  If K−1 preserves the same
phase-aligned structure as K, the extra state at K is spurious — prefer
K−1.  If K−1 merges two behaviourally distinct states, K is justified.


In [ ]:
# =============================================================================
# CELL B · K_sensitivity = K-1 sensitivity check
# =============================================================================

K_sens = K - 1   # one fewer state than the BIC-optimal model

if K_sens < 3:
    print(f"K_sens={K_sens} too small — skipping.")
else:
    print(f"Fitting K={K_sens} sensitivity model…")
    hmm_sens, logL_sens = fit_hmm_multistart(X_pca, n_states=K_sens,
                                              n_starts=5, n_iter=300)
    print(f"K={K_sens}  logL={logL_sens:.1f}  "
          f"converged={hmm_sens.monitor_.converged}")

    seq_sens_raw  = hmm_sens.predict(X_pca)
    seq_sens, _   = reorder_states_by_occupancy(seq_sens_raw, K_sens)

    occ_sens = pd.Series(seq_sens).value_counts(normalize=True).sort_index()
    print(f"\nK={K_sens} state occupancy:\n{occ_sens.round(3).to_dict()}")

    enrich_sens = behaviour_enrichment_hard(seq_sens, behaviour_masks)
    frac_sens   = enrich_sens.pivot(index="state", columns="variable",
                                    values="fraction_in_state")
    print(f"\nK={K_sens} Hard P(behaviour | state):")
    display(frac_sens.round(3))

    occ_phase_sens = state_occupancy_by_phase(seq_sens, phase_masks)
    phase_sens_tbl = (occ_phase_sens
                      .pivot(index="phase", columns="state",
                             values="occupancy_fraction")
                      .fillna(0.0))
    print(f"\nK={K_sens} Phase × state occupancy:")
    display(phase_sens_tbl.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    occ_main = pd.Series(
        reorder_states_by_occupancy(hmm_final.predict(X_pca), K)[0]
    ).value_counts(normalize=True).sort_index()

    cmap_tab = matplotlib.colormaps.get_cmap("tab10")
    axes[0].bar(range(K),      occ_main.values,
                color=[cmap_tab(s / K) for s in range(K)])
    axes[0].set_title(f"K={K}  occupancy (main model)")
    axes[0].set_xlabel("State"); axes[0].set_ylabel("Fraction")

    axes[1].bar(range(K_sens), occ_sens.values,
                color=[cmap_tab(s / K_sens) for s in range(K_sens)])
    axes[1].set_title(f"K={K_sens}  occupancy (sensitivity)")
    axes[1].set_xlabel("State"); axes[1].set_ylabel("Fraction")
    plt.tight_layout(); plt.show()

    print(f"\nInterpretation:")
    print(f"  If K={K_sens} preserves the same phase-aligned structure → "
          f"the extra state at K={K} is a split artefact; prefer K={K_sens}.")
    print(f"  If K={K_sens} merges two meaningful states → K={K} is justified.")

## 5. Decode latent state sequence

In [ ]:
# =============================================================================
# SECTION 5 · Decode state sequence
# =============================================================================

state_seq_raw  = hmm_final.predict(X_pca)
state_post_raw = hmm_final.predict_proba(X_pca)

state_seq, state_order = reorder_states_by_occupancy(state_seq_raw, K)

state_post = state_post_raw[:, [k for k, _ in
                                 sorted(state_order.items(),
                                        key=lambda x: x[1])]]

occupancy = pd.Series(state_seq).value_counts(normalize=True).sort_index()
print(f"State occupancy (0 = most occupied):\n{occupancy.round(3).to_dict()}")
print(f"\nState remapping (original → sorted): {state_order}")

cmap_states = matplotlib.colormaps.get_cmap("tab10")

## 6. Session state-sequence overview

In [ ]:
# =============================================================================
# SECTION 6 · State sequence visualisation
# =============================================================================

phase_colors = {"Pre": "tab:blue", "WhiteAgg": "tab:red",
                "DarkAgg": "gray",   "Post": "tab:green"}

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(t_min, state_seq, lw=0.7, color="black")
for phase_name, pm in phase_masks.items():
    idx = np.flatnonzero(pm)
    if len(idx):
        axes[0].axvspan(idx[0]/fs/60, idx[-1]/fs/60,
                        alpha=0.08, color=phase_colors[phase_name], label=phase_name)
axes[0].set_ylabel("HMM state")
axes[0].set_title("Latent state sequence")
axes[0].legend(loc="upper right", ncol=4, fontsize=8)

for s in range(K):
    mask_s = state_seq == s
    axes[1].fill_between(t_min, s - 0.4, s + 0.4,
                         where=mask_s, color=cmap_states(s / K), alpha=0.85)
for phase_name, pm in phase_masks.items():
    idx = np.flatnonzero(pm)
    if len(idx):
        axes[1].axvspan(idx[0]/fs/60, idx[-1]/fs/60,
                        alpha=0.06, color=phase_colors[phase_name])
axes[1].set_ylabel("State"); axes[1].set_yticks(range(K))
axes[1].set_title("State raster (one row per state)")

axes[2].stackplot(t_min, state_post.T,
                  colors=[cmap_states(s / K) for s in range(K)],
                  labels=[f"state {s}" for s in range(K)], alpha=0.85)
axes[2].set_ylabel("Posterior prob.")
axes[2].set_xlabel("Time (min)")
axes[2].set_title("Stacked state posteriors")
axes[2].legend(loc="upper right", ncol=K, fontsize=8)

plt.tight_layout(); plt.show()

## 7. State × behaviour alignment

In [ ]:
# =============================================================================
# SECTION 7 · Behaviour enrichment (hard + soft)
# =============================================================================

enrich_hard   = behaviour_enrichment_hard(state_seq, behaviour_masks)
frac_in_state = enrich_hard.pivot(index="state", columns="variable",
                                   values="fraction_in_state")
frac_of_var   = enrich_hard.pivot(index="state", columns="variable",
                                   values="fraction_of_var")

print("Hard alignment — P(behaviour | state):")
display(frac_in_state.round(3))
print("\nHard alignment — P(state | behaviour):")
display(frac_of_var.round(3))

enrich_soft = behaviour_enrichment_soft(state_post, behaviour_masks)
soft_pivot  = enrich_soft.pivot(index="state", columns="variable",
                                 values="mean_posterior_given_behaviour")
print("\nSoft alignment — mean posterior P(state | behaviour):")
display(soft_pivot.round(3))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, data, title in [
    (axes[0], frac_in_state, "Hard: P(behaviour | state)"),
    (axes[1], frac_of_var,   "Hard: P(state | behaviour)"),
    (axes[2], soft_pivot,    "Soft: P(state | behaviour)\n[posterior-weighted]"),
]:
    if data is not None and len(data):
        im = ax.imshow(data.values.astype(float), aspect="auto", cmap="Blues",
                       vmin=0, vmax=data.values.astype(float).max())
        ax.set_xticks(range(len(data.columns)))
        ax.set_xticklabels(data.columns, rotation=35, ha="right", fontsize=9)
        ax.set_yticks(range(len(data.index)))
        ax.set_yticklabels([f"state {s}" for s in data.index])
        ax.set_title(title); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## 8. State occupancy by phase & Pre → Post bootstrap test

In [ ]:
# =============================================================================
# SECTION 8 · Phase occupancy + Pre→Post bootstrap test
# =============================================================================

occ_phase_df   = state_occupancy_by_phase(state_seq, phase_masks)
phase_occ_table = (occ_phase_df
                   .pivot(index="phase", columns="state",
                          values="occupancy_fraction")
                   .fillna(0.0))
display(phase_occ_table.round(3))

print("\nBootstrap 95% CI for (Post − Pre) occupancy difference (n_boot=1000):")
prepost_df = bootstrap_prepost_diff(state_seq, phase_masks, K,
                                    n_boot=1000, seed=42)
display(prepost_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

phase_occ_table.plot(kind="bar", stacked=True, ax=axes[0],
                     colormap="tab10", alpha=0.85)
axes[0].set_ylabel("Occupancy fraction"); axes[0].set_xlabel("Phase")
axes[0].set_title("HMM state occupancy by phase")
axes[0].legend(title="state", bbox_to_anchor=(1.02, 1),
               loc="upper left", fontsize=8)
axes[0].tick_params(axis="x", rotation=20)

for s in range(K):
    row = prepost_df[prepost_df["state"] == s].iloc[0]
    col = cmap_states(s / K)
    axes[1].bar(s, row["obs_diff"], color=col, alpha=0.8)
    axes[1].errorbar(s, row["obs_diff"],
                     yerr=[[row["obs_diff"] - row["ci_lo"]],
                           [row["ci_hi"]   - row["obs_diff"]]],
                     fmt="none", color="black", capsize=4)
    if row["significant"]:
        axes[1].text(s, row["ci_hi"] + 0.005, "*", ha="center", fontsize=12)
axes[1].axhline(0, color="k", lw=1)
axes[1].set_xticks(range(K))
axes[1].set_xticklabels([f"state {s}" for s in range(K)], rotation=30, ha="right")
axes[1].set_ylabel("Post − Pre occupancy fraction")
axes[1].set_title("Pre → Post occupancy change\n(bootstrap 95% CI; * = CI excludes 0)")
plt.tight_layout(); plt.show()

## 9. Transition structure and dwell times

In [ ]:
# =============================================================================
# SECTION 9 · Transition matrix + dwell times
# =============================================================================

new_order_keys = [k for k, _ in sorted(state_order.items(), key=lambda x: x[1])]
transmat_df = pd.DataFrame(
    hmm_final.transmat_[np.ix_(new_order_keys, new_order_keys)],
    index=[f"state_{i}" for i in range(K)],
    columns=[f"state_{i}" for i in range(K)],
)
display(transmat_df.round(3))

dwell_df, dwell_summary = state_dwell_summary(state_seq, fs=fs)
print("\nDwell-time summary (s):")
display(dwell_summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im = axes[0].imshow(transmat_df.values, aspect="auto", cmap="Blues", vmin=0, vmax=1)
axes[0].set_xticks(range(K)); axes[0].set_xticklabels([f"s{i}" for i in range(K)])
axes[0].set_yticks(range(K)); axes[0].set_yticklabels([f"s{i}" for i in range(K)])
axes[0].set_xlabel("To"); axes[0].set_ylabel("From")
axes[0].set_title("Transition matrix")
plt.colorbar(im, ax=axes[0])

self_trans = np.diag(transmat_df.values)
axes[1].bar(range(K), self_trans,
            color=[cmap_states(s / K) for s in range(K)], alpha=0.85)
axes[1].set_xticks(range(K))
axes[1].set_xticklabels([f"state {s}" for s in range(K)], rotation=30, ha="right")
axes[1].set_ylim(0, 1); axes[1].set_ylabel("Self-transition probability")
axes[1].set_title("State persistence")

axes[2].bar(dwell_summary["state"].astype(str), dwell_summary["mean"],
            color=[cmap_states(s / K) for s in dwell_summary["state"]], alpha=0.85)
axes[2].set_xlabel("State"); axes[2].set_ylabel("Mean dwell time (s)")
axes[2].set_title("Mean dwell time by latent state")

plt.tight_layout(); plt.show()

## 10. HMM emission means — what does each state look like neurally?

In [ ]:
# =============================================================================
# SECTION 10 · Emission mean visualisation
# =============================================================================

means_orig   = hmm_final.means_
means_scaled = means_orig[new_order_keys]
means_pca_raw = scaler.inverse_transform(means_scaled)
means_neural  = pca_hmm.inverse_transform(means_pca_raw)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

neuron_order = np.argsort(pca_hmm.components_[0])
im0 = axes[0].imshow(means_neural[:, neuron_order].T,
                      aspect="auto", cmap="RdBu_r",
                      vmin=-np.abs(means_neural).max(),
                      vmax=np.abs(means_neural).max())
axes[0].set_xticks(range(K))
axes[0].set_xticklabels([f"state {s}" for s in range(K)],
                         rotation=30, ha="right")
axes[0].set_ylabel("Neuron (sorted by PC1 loading)")
axes[0].set_title("HMM emission means in neural z-score space")
plt.colorbar(im0, ax=axes[0], label="z-score")

n_show = min(5, N_PCS)
for s in range(K):
    axes[1].plot(range(1, n_show + 1), means_pca_raw[s, :n_show],
                 marker="o", label=f"state {s}", color=cmap_states(s / K))
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].set_xlabel("PC")
axes[1].set_ylabel("Emission mean (unstandardised PC score)")
axes[1].set_title(f"Emission means along first {n_show} PCs")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 11. HMM states projected into supervised summary space

In [ ]:
# =============================================================================
# SECTION 11 · Overlay: HMM states + supervised conditions in shared PCA space
# =============================================================================

supervised_masks = {
    "Pre_baseline":  phase_masks["Pre"]  & (~sniff_urine) & (~lick_milk) & (~lick_empty),
    "Pre_sniff":     phase_masks["Pre"]  & sniff_urine,
    "WA":            phase_masks["WhiteAgg"] & attack,
    "WN":            phase_masks["WhiteAgg"] & (~attack),
    "W_sub":         phase_masks["WhiteAgg"] & submission,
    "DA":            phase_masks["DarkAgg"]  & attack,
    "DN":            phase_masks["DarkAgg"]  & (~attack),
    "Post_sniff":    phase_masks["Post"] & sniff_urine,
    "Post_baseline": phase_masks["Post"] & (~sniff_urine) & (~lick_milk) & (~lick_empty),
}
valid_sup = {k: v for k, v in supervised_masks.items()
             if int(np.asarray(v).astype(bool).sum()) > 10}

pca_shared = PCA(n_components=3, random_state=0)
pca_shared.fit(X_full)

X_sup  = np.stack([condition_pattern(FRz, v) for v in valid_sup.values()], axis=0)
Z_sup  = pca_shared.transform(X_sup)

state_cent_neural = np.stack(
    [FRz[:, state_seq == s].mean(axis=1) for s in range(K)], axis=0)
Z_hmm = pca_shared.transform(state_cent_neural)

print("Shared PCA explained variance (PC1-3):",
      np.round(pca_shared.explained_variance_ratio_, 4))

sup_markers = {"Pre_baseline": "o", "Pre_sniff": "^",
               "WA": "D", "WN": "s", "W_sub": "P",
               "DA": "D", "DN": "s",
               "Post_sniff": "^", "Post_baseline": "o"}
sup_colors  = {"Pre_baseline": "cornflowerblue", "Pre_sniff": "tab:blue",
               "WA": "tab:red",       "WN": "lightcoral",
               "W_sub": "tab:purple",
               "DA": "darkorange",    "DN": "moccasin",
               "Post_sniff": "mediumseagreen", "Post_baseline": "tab:green"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax_idx, (pc_x, pc_y) in enumerate([(0, 1), (0, 2)]):
    ax = axes[ax_idx]
    for i, nm in enumerate(valid_sup.keys()):
        ax.scatter(Z_sup[i, pc_x], Z_sup[i, pc_y],
                   s=120, color=sup_colors.get(nm, "gray"),
                   marker=sup_markers.get(nm, "o"), zorder=5, label=nm)
        ax.annotate(nm, (Z_sup[i, pc_x], Z_sup[i, pc_y]),
                    textcoords="offset points", xytext=(6, 4), fontsize=8)
    for s in range(K):
        ax.scatter(Z_hmm[s, pc_x], Z_hmm[s, pc_y],
                   s=200, color=cmap_states(s / K), marker="X",
                   edgecolors="black", linewidths=0.8, zorder=6)
        ax.annotate(f"HMM-{s}", (Z_hmm[s, pc_x], Z_hmm[s, pc_y]),
                    textcoords="offset points", xytext=(6, -8),
                    fontsize=9, fontweight="bold", color=cmap_states(s / K))
    ax.axhline(0, lw=0.4, color="gray"); ax.axvline(0, lw=0.4, color="gray")
    ax.set_xlabel(f"PC{pc_x+1}"); ax.set_ylabel(f"PC{pc_y+1}")
    ax.set_title(f"HMM states (✕) + supervised conditions\n"
                 f"shared neural PCA (PC{pc_x+1} vs PC{pc_y+1})")
plt.tight_layout(); plt.show()

sup_hmm_dist = cdist(Z_sup[:, :2], Z_hmm[:, :2], metric="euclidean")
sup_hmm_df   = pd.DataFrame(sup_hmm_dist,
                             index=list(valid_sup.keys()),
                             columns=[f"HMM-{s}" for s in range(K)])
sup_hmm_df["nearest_HMM_state"] = sup_hmm_df.idxmin(axis=1)
print("\nNearest HMM state to each supervised condition (PC1-2 Euclidean):")
display(sup_hmm_df.round(3))

## 12. Attack-aligned latent-state occupancy (PETH)

In [ ]:
# =============================================================================
# SECTION 12 · Attack-aligned state PETH
# =============================================================================

pre_sec   = 2.0; post_sec = 4.0
pre_bins  = int(pre_sec * fs); post_bins = int(post_sec * fs)
nT        = len(state_seq)
time_event = np.arange(-pre_bins, post_bins + 1) / fs

all_onsets = np.flatnonzero(
    np.diff(np.concatenate([[False], attack]).astype(int)) == 1)

def filter_onsets(onsets, phase_mask, nT, pre_b, post_b):
    return np.array([o for o in onsets.astype(int)
                     if pre_b <= o < nT - post_b and phase_mask[o]], dtype=int)

white_onsets = filter_onsets(all_onsets, phase_masks["WhiteAgg"], nT,
                              pre_bins, post_bins)
dark_onsets  = filter_onsets(all_onsets, phase_masks["DarkAgg"],  nT,
                              pre_bins, post_bins)
print(f"Attack onsets — White: {len(white_onsets)},  Dark: {len(dark_onsets)}")

hard_peth_w, soft_peth_w = attack_state_peth(
    state_seq, state_post, K, white_onsets, pre_bins, post_bins)
hard_peth_d, soft_peth_d = attack_state_peth(
    state_seq, state_post, K, dark_onsets,  pre_bins, post_bins)

fig, axes = plt.subplots(K, 2, figsize=(13, 2.0 * K), sharex=True, sharey="row")
fig.suptitle("Attack-aligned state occupancy (baseline-corrected)\n"
             "Left = Hard (Viterbi),  Right = Soft (posterior)", fontsize=11)
for s in range(K):
    for col, peth_w, peth_d, label in [
        (0, hard_peth_w, hard_peth_d, "Hard"),
        (1, soft_peth_w, soft_peth_d, "Soft"),
    ]:
        ax = axes[s, col]
        ax.plot(time_event, peth_w[s], lw=2, label="White", color="steelblue")
        ax.plot(time_event, peth_d[s], lw=2, label="Dark",  color="darkorange")
        ax.axvline(0, color="k", lw=1, ls="--")
        ax.axhline(0, color="k", lw=0.5)
        ax.set_ylabel(f"state {s}\nΔ occ.", fontsize=8)
        if s == 0: ax.set_title(label)
        if s == 0: ax.legend(fontsize=8)
axes[-1, 0].set_xlabel("Time from attack onset (s)")
axes[-1, 1].set_xlabel("Time from attack onset (s)")
plt.tight_layout(); plt.show()

## 12B. State 6 deep-dive — highest attack-enriched state

State 6 has the highest attack enrichment (attack=18.1%) and spans both White
and Dark phases. It is the most direct bridge between the HMM and the supervised
notebook's WA/DA conditions.

This cell quantifies:
1. Centroid distances between state 6 and WA, WN, DA, DN
2. What fraction of WA/DA bins fall inside state 6
3. Absolute + baseline-corrected attack PETH for state 6 alone
4. Where state 6 sits in the supervised PCA space

In [ ]:
# =============================================================================
# CELL C · State 6 deep-dive
# =============================================================================

# ── Identify the attack-enriched state ──────────────────────────────────────
# Default is state index 6 based on the K=10 results; update if K changed.
# We compute it dynamically from the enrichment table.
if "frac_in_state" in dir() and frac_in_state is not None and len(frac_in_state):
    attack_enrichment_by_state = frac_in_state["attack"]
    atk_state = int(attack_enrichment_by_state.idxmax())
    print(f"Most attack-enriched state: {atk_state}  "
          f"(attack fraction = {attack_enrichment_by_state.max():.3f})")
else:
    atk_state = 6
    print(f"Using default atk_state = {atk_state} (frac_in_state not available)")

# ── 1. Centroid distances vs WA / WN / DA / DN ──────────────────────────────
supervised_attack_masks = {
    "WA": phase_masks["WhiteAgg"] & attack,
    "WN": phase_masks["WhiteAgg"] & (~attack),
    "DA": phase_masks["DarkAgg"]  & attack,
    "DN": phase_masks["DarkAgg"]  & (~attack),
}

atk_state_centroid = condition_pattern(FRz, state_seq == atk_state)
sup_atk_centroids  = {nm: condition_pattern(FRz, m)
                      for nm, m in supervised_attack_masks.items()}
sup_atk_centroids[f"HMM-s{atk_state}"] = atk_state_centroid

eu_atk, cos_atk = pairwise_distance_tables(sup_atk_centroids)
print(f"\nState {atk_state} vs supervised attack conditions — Euclidean distances:")
display(eu_atk.round(3))
print(f"\nCosine similarities:")
display(cos_atk.round(3))

# ── 2. What fraction of WA/DA bins land in this state? ──────────────────────
print(f"\nFraction of supervised attack bins assigned to state {atk_state}:")
for nm, mask in supervised_attack_masks.items():
    in_s = (state_seq == atk_state) & mask
    pct  = 100 * in_s.sum() / max(mask.sum(), 1)
    print(f"  {nm}: {pct:.1f}%  ({in_s.sum()} / {mask.sum()} bins)")

# ── 3. Focused PETH for this state (raw + baseline-corrected) ───────────────
hard_s_w_raw, soft_s_w_raw = raw_peth_single_state(
    state_seq, state_post, atk_state, white_onsets, pre_bins, post_bins)
hard_s_d_raw, soft_s_d_raw = raw_peth_single_state(
    state_seq, state_post, atk_state, dark_onsets,  pre_bins, post_bins)

bl_w = hard_s_w_raw[:pre_bins].mean()
bl_d = hard_s_d_raw[:pre_bins].mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Raw occupancy
axes[0].plot(time_event, hard_s_w_raw, lw=2, label="White", color="steelblue")
axes[0].plot(time_event, hard_s_d_raw, lw=2, label="Dark",  color="darkorange")
axes[0].axvline(0, color="k", lw=1, ls="--")
axes[0].set_xlabel("Time from attack onset (s)")
axes[0].set_ylabel(f"P(state {atk_state})")
axes[0].set_title(f"State {atk_state} — absolute occupancy\naround attack onset")
axes[0].legend()

# Baseline-corrected
axes[1].plot(time_event, hard_s_w_raw - bl_w, lw=2, label="White", color="steelblue")
axes[1].plot(time_event, hard_s_d_raw - bl_d, lw=2, label="Dark",  color="darkorange")
axes[1].axvline(0, color="k", lw=1, ls="--")
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_xlabel("Time from attack onset (s)")
axes[1].set_ylabel(f"ΔP(state {atk_state})")
axes[1].set_title(f"State {atk_state} — baseline-corrected\n"
                   f"(White vs Dark asymmetry)")
axes[1].legend()

# PCA overlay (2 PCs for clarity)
pca_atk = PCA(n_components=2, random_state=0)
pca_atk.fit(X_full)

labels_sup_atk = list(supervised_attack_masks.keys())
X_sup_atk      = np.stack([sup_atk_centroids[nm] for nm in labels_sup_atk], axis=0)
Z_sup_atk      = pca_atk.transform(X_sup_atk)
Z_s_atk        = pca_atk.transform(atk_state_centroid[None, :])

cols_atk = {"WA": "tab:red", "WN": "lightcoral",
            "DA": "darkorange", "DN": "moccasin"}
for i, nm in enumerate(labels_sup_atk):
    axes[2].scatter(Z_sup_atk[i, 0], Z_sup_atk[i, 1],
                    s=120, color=cols_atk[nm], zorder=5)
    axes[2].annotate(nm, (Z_sup_atk[i, 0], Z_sup_atk[i, 1]),
                     xytext=(5, 4), textcoords="offset points", fontsize=9)
axes[2].scatter(Z_s_atk[0, 0], Z_s_atk[0, 1],
                s=220, color="tab:purple", marker="X",
                edgecolors="black", linewidths=0.8, zorder=6)
axes[2].annotate(f"HMM-s{atk_state}",
                 (Z_s_atk[0, 0], Z_s_atk[0, 1]),
                 xytext=(5, -10), textcoords="offset points",
                 fontsize=10, fontweight="bold", color="tab:purple")
axes[2].axhline(0, lw=0.4, color="gray"); axes[2].axvline(0, lw=0.4, color="gray")
axes[2].set_xlabel("PC1"); axes[2].set_ylabel("PC2")
axes[2].set_title(f"State {atk_state} centroid (✕) vs WA/WN/DA/DN\n"
                   f"in shared neural PCA space")
plt.tight_layout(); plt.show()

# ── 4. Summary ───────────────────────────────────────────────────────────────
nearest_sup = eu_atk.loc[f"HMM-s{atk_state}"].drop(f"HMM-s{atk_state}").idxmin()
mean_delta_w = (hard_s_w_raw[pre_bins: pre_bins + int(1 * fs)] - bl_w).mean()
mean_delta_d = (hard_s_d_raw[pre_bins: pre_bins + int(1 * fs)] - bl_d).mean()

print(f"\nState {atk_state} summary:")
print(f"  Overall occupancy:          {(state_seq == atk_state).mean():.3f}")
for phase_name, pm in phase_masks.items():
    frac = (state_seq[pm] == atk_state).mean()
    print(f"  {phase_name:12s} occupancy: {frac:.3f}")
print(f"  Nearest supervised condition:           {nearest_sup}")
print(f"  Attack Δ-occ 0–1s (White): {mean_delta_w:+.4f}")
print(f"  Attack Δ-occ 0–1s (Dark):  {mean_delta_d:+.4f}")

## 13. Compact results summary

In [ ]:
# =============================================================================
# SECTION 13 · Summary tables
# =============================================================================

print("=" * 65)
print("MODEL SELECTION")
display(model_sel_df.round(2))

print("\n" + "=" * 65)
print(f"CHOSEN K = {K}  |  N_PCS = {N_PCS}  |  "
      f"converged = {hmm_final.monitor_.converged}  |  "
      f"logL = {hmm_final.score(X_pca):.1f}")

print("\n" + "=" * 65)
print("STATE OCCUPANCY FRACTIONS")
display(occupancy.to_frame("fraction").T.round(3))

print("\n" + "=" * 65)
print("DWELL TIME SUMMARY (s)")
display(dwell_summary.round(3))

print("\n" + "=" * 65)
print("PHASE × STATE OCCUPANCY")
display(phase_occ_table.round(3))

print("\n" + "=" * 65)
print("Pre → Post OCCUPANCY CHANGE (bootstrap 95% CI)")
display(prepost_df.round(4))

print("\n" + "=" * 65)
print("BEHAVIOUR ENRICHMENT — Hard P(behaviour | state)")
display(frac_in_state.round(3))

print("\n" + "=" * 65)
print("BEHAVIOUR ENRICHMENT — Soft P(state | behaviour)")
display(soft_pivot.round(3))

print("\n" + "=" * 65)
print("NEAREST SUPERVISED CONDITION TO EACH HMM STATE")
display(sup_hmm_df[["nearest_HMM_state"]].T)

print("\n" + "=" * 65)
print(f"ATTACK-ENRICHED STATE (state {atk_state}) vs SUPERVISED CONDITIONS")
display(eu_atk.round(3))

## 14. Discussion

### What this notebook adds to the supervised analysis

| Supervised geometry | HMM latent states |
|---|---|
| Behaviour labels imposed first | No labels — states emerge from population |
| Decoding accuracy quantifies separability | Occupancy + transition structure characterises dynamics |
| Centroid geometry in z-score / PCA space | State sequence with temporal persistence |
| Cross-phase generalisation | Phase-specific occupancy changes with bootstrap CIs |

### How to read the key results

**Cell A (N_PCS sweep)** — the most important methodological check.
If BIC/AIC still fall monotonically at N_PCS=20, increase to 30 and extend
the K range. The goal is to find the smallest N_PCS where an elbow exists.

**Cell B (K sensitivity)** — verifies that the least-occupied state is not a
split artefact. If K−1 produces the same phase-aligned structure, prefer it
for a cleaner story.

**Behaviour enrichment heatmap** — the primary scientific output. Phase-aligned
states that also show attack, submission, or sniff enrichment are the most
interesting: the HMM found them without any behavioural supervision.

**Pre → Post bootstrap test** — converging evidence for session remapping.
States with significant (Post − Pre) differences that also correspond to
supervised Pre/Post conditions strengthen the remapping conclusion.

**Cell C (state deep-dive)** — the bridge between notebooks. If the most
attack-enriched HMM state sits close to WA/DA in the shared PCA space,
the two analyses tell the same story from different angles.

### Caveats
- A Gaussian HMM is a first model; no memory beyond first-order Markov.
- State labels are arbitrary; occupancy ordering is a convenience only.
- BIC/AIC are heuristics — autocorrelation means effective N << 60,000 bins.
- Multi-start reduces but does not eliminate local optima.
- Single-session exploratory analysis.

### Why BIC fails and how K is selected

**BIC failure mode.**  Raw BIC uses log(n_timebins) = log(60,000) ≈ 11.0
as the per-parameter penalty.  For a Gaussian HMM on 218-neuron population
data, each additional state captures substantial genuine variance, so the
per-state log-likelihood gain (~30,000–50,000) dwarfs any plausible BIC
penalty.  Replacing n with n_eff = n / τ_detrend (phase-detrended
integrated autocorrelation time, Section 3B) reduces the penalty but does
not restore a finite minimum — BIC and its corrected variant both remain
monotone through K = 15.  This is expected for high-dimensional,
autocorrelated, multi-modal neural data, where BIC's i.i.d. likelihood
assumption is severely violated.

**Temporal cross-validation** (Section 4) provides a principled alternative:
the session is split into 5 contiguous folds; each fold is held out in turn,
and the model trained on the remaining 4 folds is scored on the held-out
window.  The held-out log-likelihood per bin plateaus after the CV elbow,
identifying the K beyond which additional states no longer generalise.

**State stability (ARI)** (Section 4) guards against over-fitting: if the
same K fitted with different random seeds produces inconsistent Viterbi
sequences (mean pairwise ARI < 0.3), the solution is not reproducible and K
should be reduced.

**Selection rule**: K = min(K_cv, K_ari_max), where K_cv is the CV elbow
and K_ari_max is the largest K with ARI ≥ 0.3.


The raw BIC decreases monotonically through K = 15 because n_timebins = 60 000
inflates the log-penalty (log 60 000 ≈ 11.0) relative to the true information
content.  Neural spike-rate traces are strongly autocorrelated; adjacent 20 ms
bins are far from independent.  Replacing n by n_eff = n / τ_int (where τ_int
is the Sokal-windowed integrated autocorrelation time estimated from PC1)
produces a corrected BIC with a well-defined minimum.  The τ sensitivity sweep
confirms that the elbow is stable across τ ∈ {50, 100, 200, 500} bins,
spanning the plausible range of neural autocorrelation timescales in this
dataset.  This correction is conceptually equivalent to the effective-sample-
size correction used in time-series cross-validation and MCMC diagnostics
(Thiébaux & Zwiers 1984; Geyer 1992).
